In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.metrics import (accuracy_score,f1_score,roc_auc_score,confusion_matrix,classification_report)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
!pip install xgboost
from xgboost import XGBClassifier


from sklearn.model_selection import cross_val_score

import warnings
warnings.filterwarnings('ignore')

In [2]:
TW = pd.read_csv(
    r"../classification/Twitter/Absolute_labeling/Twitter-Absolute-Sigma-500.data",
    sep=",",
    header=None
)

groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label")  # остання колонка

TW.columns = columns

TW.head(2)

,NCD_0,NCD_1,NCD_2,NCD_3,NCD_4,NCD_5,NCD_6,AI_0,AI_1,AI_2,...,ADL_5,ADL_6,NAD_0,NAD_1,NAD_2,NAD_3,NAD_4,NAD_5,NAD_6,label
0,889,939,960,805,805,1143,1121,549,613,587,...,1.0,1.0,889,939,960,805,805,1143,1121,1.0
1,542,473,504,626,647,795,832,366,288,318,...,1.0,1.0,542,473,504,626,647,795,832,1.0


In [3]:
TH = pd.read_csv(
    "../classification/TomsHardware/Absolute_labeling/TomsHardware-Absolute-Sigma-500.data",
    sep=",",
    header=None
)


groups = [
    "NCD", "BL", "NAD", "AI", "NAC", "ND",
    "CS", "AT", "NA", "ADL", "AS_NA", "AS_NAC"
]

columns = []
for group in groups:
    for t in range(8):
        columns.append(f"{group}_{t}")

columns.append("label")  # остання колонка

TH.columns = columns


prefixes = {col.split("_")[0] for col in TH.columns if "_" in col}


TH.head()

,NCD_0,NCD_1,NCD_2,NCD_3,NCD_4,NCD_5,NCD_6,NCD_7,BL_0,BL_1,...,AS_NA_7,AS_NAC_0,AS_NAC_1,AS_NAC_2,AS_NAC_3,AS_NAC_4,AS_NAC_5,AS_NAC_6,AS_NAC_7,label
0,1,0,0,0,0,0,0,1,1.0,0.0,...,0.001816,0.001211,0.000560,0.000000,0.000000,0.000161,0.0,0.000301,0.000818,1.0
1,1,1,1,1,0,0,0,0,1.0,1.0,...,0.005029,0.000784,0.000802,0.001592,0.001612,0.000741,0.0,0.000545,0.002437,1.0
2,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
3,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
4,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0


### Twitter

In [16]:
X = TW.drop(columns=["label"])
y = TW["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

### SVM without scaling

In [6]:

svm = SVC(random_state=42)  

svm.fit(X_train, y_train)

pred_svm = svm.predict(X_test)

accuracy = accuracy_score(y_test, pred_svm)
precision = precision_score(y_test, pred_svm, zero_division=0)
recall = recall_score(y_test, pred_svm, zero_division=0)
f1 = f1_score(y_test, pred_svm, zero_division=0)

print("SVM baseline")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")

SVM baseline
Accuracy: 0.9688010802359462
Precision: 0.9406444318824194
Recall: 0.8986498649864987
F1-score: 0.9191677407475602


### SVM wirh Scaling and class_weight

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = SVC( kernel='linear',class_weight='balanced',probability=True, random_state=42)

svm.fit(X_train_scaled, y_train)


pred_svm = svm.predict(X_test_scaled)
pred_proba = svm.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, pred_svm)
precision = precision_score(y_test, pred_svm, zero_division=0)
recall = recall_score(y_test, pred_svm, zero_division=0)
f1 = f1_score(y_test, pred_svm, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("SVM wirh Scaling and class_weight")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

SVM wirh Scaling and class_weight
Accuracy: 0.9562575509914008
Precision: 0.8397234443746072
Recall: 0.962016201620162
F1-score: 0.8967195234499539
ROC-AUC: 0.9923133181959449


### SVM Grid Search

In [17]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(random_state=42))
])

param_grid = {
    'model__C':  [1, 10],
    'model__kernel': ['linear'],  
    'model__class_weight': [None, 'balanced']
}


grid = GridSearchCV(pipeline,param_grid,cv=2,scoring='f1',n_jobs=-1)

grid.fit(X_train, y_train)

best_svm = grid.best_estimator_

pred_svm = best_svm.predict(X_test)

accuracy = accuracy_score(y_test, pred_svm)
precision = precision_score(y_test, pred_svm, zero_division=0)
recall = recall_score(y_test, pred_svm, zero_division=0)
f1 = f1_score(y_test, pred_svm, zero_division=0)

print("SVM Grid Search")
print("Best params:", grid.best_params_)
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")

SVM Grid Search
Best params: {'model__C': 1, 'model__class_weight': None, 'model__kernel': 'linear'}
Accuracy: 0.968197000923886
Precision: 0.9401208915753684
Recall: 0.895949594959496
F1-score: 0.917503917411743


In [19]:
import pandas as pd

data = {
    "Model": [
        "SVM",
        "SVM with Scaling and class_weight",
        "SVM with Grid Search"
    ],
    "Accuracy": [0.9688, 0.9563, 0.9682],
    "Precision": [0.9406, 0.8397, 0.9401],
    "Recall": [0.8986, 0.9620, 0.8959],
    "F1-score": [0.9192, 0.8967, 0.9175]
}

df_svm_tw = pd.DataFrame(data)
df_svm_tw

,Model,Accuracy,Precision,Recall,F1-score
0,SVM,0.9688,0.9406,0.8986,0.9192
1,SVM with Scaling and class_weight,0.9563,0.8397,0.9620,0.8967
2,SVM with Grid Search,0.9682,0.9401,0.8959,0.9175


### Tom Harwer

In [10]:
X = TH.drop(columns=["label"])
y = TH["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

### SVM without scaling

In [11]:

svm = SVC(random_state=42)  

svm.fit(X_train, y_train)

pred_svm = svm.predict(X_test)

accuracy = accuracy_score(y_test, pred_svm)
precision = precision_score(y_test, pred_svm, zero_division=0)
recall = recall_score(y_test, pred_svm, zero_division=0)
f1 = f1_score(y_test, pred_svm, zero_division=0)

print("SVM baseline")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")

SVM baseline
Accuracy: 0.9000632511068943
Precision: 0.9963414634146341
Recall: 0.8405349794238683
F1-score: 0.9118303571428571


### SVM wirh Scaling and class_weight

In [12]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = SVC( kernel='linear',class_weight='balanced',probability=True, random_state=42)

svm.fit(X_train_scaled, y_train)


pred_svm = svm.predict(X_test_scaled)
pred_proba = svm.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, pred_svm)
precision = precision_score(y_test, pred_svm, zero_division=0)
recall = recall_score(y_test, pred_svm, zero_division=0)
f1 = f1_score(y_test, pred_svm, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("SVM wirh Scaling and class_weight")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

SVM wirh Scaling and class_weight
Accuracy: 0.9259962049335864
Precision: 0.9819616685456595
Recall: 0.8960905349794238
F1-score: 0.9370629370629371
ROC-AUC: 0.9893216971761033


### SVM Grid Search

In [13]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(random_state=42))
])

param_grid = {
    'model__C': [0.1, 1, 10],
    'model__kernel': ['linear'],  
    'model__class_weight': [None, 'balanced']
}


grid = GridSearchCV(pipeline,param_grid,cv=3,scoring='f1',n_jobs=-1)

grid.fit(X_train, y_train)

best_svm = grid.best_estimator_

pred_svm = best_svm.predict(X_test)

accuracy = accuracy_score(y_test, pred_svm)
precision = precision_score(y_test, pred_svm, zero_division=0)
recall = recall_score(y_test, pred_svm, zero_division=0)
f1 = f1_score(y_test, pred_svm, zero_division=0)

print("SVM Grid Search")
print("Best params:", grid.best_params_)
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")

SVM Grid Search
Best params: {'model__C': 10, 'model__class_weight': None, 'model__kernel': 'linear'}
Accuracy: 0.9557242251739405
Precision: 0.9717573221757322
Recall: 0.9557613168724279
F1-score: 0.9636929460580913


In [20]:
import pandas as pd

data = {
    "Model": [
        "SVM",
        "SVM with Scaling and class_weight",
        "SVM with Grid Search"
    ],
    "Accuracy": [0.9001, 0.9260, 0.9557],
    "Precision": [0.9963, 0.9820, 0.9718],
    "Recall": [0.8405, 0.8961, 0.9558],
    "F1-score": [0.9118, 0.9371, 0.9637],
    "ROC-AUC": [None, 0.9893, None]
}

df_svm_th = pd.DataFrame(data)
df_svm_th

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,SVM,0.9001,0.9963,0.8405,0.9118,NaN
1,SVM with Scaling and class_weight,0.9260,0.9820,0.8961,0.9371,0.9893
2,SVM with Grid Search,0.9557,0.9718,0.9558,0.9637,NaN
